# 02. Chuẩn hoá và ghép 6 nguồn dữ liệu
Mỗi nguồn được chuyển về **một cấu trúc cột chung**, cùng đơn vị (giá: VNĐ, diện tích: m², chiều dài: m), cùng cách ghi địa danh và loại bất động sản. Các hàm xử lý chuỗi dùng chung nằm trong `src/xu_ly_chuoi.py`. Chưa loại bỏ dòng nào ở bước này, kể cả dòng lỗi (xử lý ở notebook 03).

In [1]:
import sys, os, re, time
sys.path.append('../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xu_ly_chuoi import *

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 60)
sns.set_theme(style='whitegrid')
RAW = '../data/raw'

INTERIM = '../data/interim'
os.makedirs(INTERIM, exist_ok=True)
COT = ['nguon', 'ma_tin', 'tieu_de', 'loai_bds', 'vi_tri', 'tinh', 'tinh_cu', 'dia_gioi_moi', 'quan_huyen', 'phuong_xa',
       'duong', 'du_an', 'gia_vnd', 'gia_trich_xuat', 'dien_tich_m2', 'dien_tich_trich_xuat', 'mat_tien_m', 'chieu_dai_m',
       'duong_vao_m', 'so_tang', 'so_phong_ngu', 'so_phong_tam', 'huong_nha', 'phap_ly', 'noi_that', 'vi_do', 'kinh_do',
       'ngay_dang']

### Các hàm chuẩn hoá giá trị phân loại

In [2]:
def chuan_huong(s):
    if pd.isna(s) or str(s).strip() in ('', '0'):
        return np.nan
    t = re.sub(r'[\s\-]+', ' ', str(s)).strip().title()
    return t if t in {'Đông', 'Tây', 'Nam', 'Bắc', 'Đông Nam', 'Đông Bắc', 'Tây Nam', 'Tây Bắc'} else np.nan

def chuan_phap_ly(s):
    if pd.isna(s) or str(s).strip() in ('', '0'):
        return np.nan
    t = bo_dau(s).lower()
    if re.search(r'hop dong|sale contract|hdmb', t):
        return 'Hợp đồng mua bán'
    if re.search(r'cho so|dang lam so|sap co so', t):
        return 'Đang chờ sổ'
    if re.search(r'so do|so hong|co so|certificate|shr|sdcc|so rieng', t):
        return 'Có sổ'
    return 'Khác'

def chuan_noi_that(s):
    if pd.isna(s) or str(s).strip() in ('', '0'):
        return np.nan
    t = bo_dau(s).lower()
    if re.search(r'full|day du|cao cap|dong bo|sang trong', t):
        return 'Đầy đủ'
    if re.search(r'basic|co ban', t):
        return 'Cơ bản'
    if re.search(r'tho|khong noi that|ban giao hoan thien mat ngoai', t):
        return 'Không / thô'
    return np.nan

def loai_tu_tieu_de(t):
    if pd.isna(t):
        return np.nan
    t = bo_dau(t).lower()
    if re.search(r'can ho|chung cu|\bccmn\b|\bcc mini\b|penthouse|duplex|studio', t):
        return 'Căn hộ'
    if re.search(r'shophouse|nha pho thuong mai', t):
        return 'Shophouse'
    if re.search(r'biet thu|lien ke|villa', t):
        return 'Biệt thự/Liền kề'
    if re.search(r'\bkho\b|nha xuong|\bxuong\b', t):
        return 'Kho/Xưởng'
    if re.search(r'\bdat\b|lo dat|dat nen|tho cu', t) and not re.search(r'\bnha\b', t):
        return 'Đất'
    return 'Nhà ở'

print([chuan_huong(x) for x in ['Đông - Nam', 'Tây Bắc', 'tây-nam', '0']])
print([chuan_phap_ly(x) for x in ['Đã có sổ', 'Sổ đỏ/ Sổ hồng', 'Have certificate', 'Sale contract', 'Đang chờ sổ']])
print([loai_tu_tieu_de(x) for x in ['Bán căn hộ 2PN Vinhomes', 'Bán lô đất thổ cư 100m2', 'Bán nhà 4 tầng ngõ ô tô', 'Biệt thự song lập']])

['Đông Nam', 'Tây Bắc', 'Tây Nam', nan]
['Có sổ', 'Có sổ', 'Có sổ', 'Hợp đồng mua bán', 'Đang chờ sổ']
['Căn hộ', 'Đất', 'Nhà ở', 'Biệt thự/Liền kề']


### Nguồn 1 (1 triệu dòng) - đọc theo từng khối 200.000 dòng, trích giá và diện tích từ tiêu đề + mô tả rồi bỏ cột mô tả để tiết kiệm bộ nhớ

In [3]:
LOAI_1 = {'Nhà': 'Nhà ở', 'Đất': 'Đất', 'Căn hộ chung cư': 'Căn hộ', 'Biệt thự/Nhà liền kề': 'Biệt thự/Liền kề', 'Shophouse': 'Shophouse'}

def doc_nguon1():
    ket_qua = []
    for khoi in pd.read_csv(os.path.join(RAW, 'nguon1_vietnam_real_estate_2025.csv'), chunksize=200000, low_memory=False):
        g_tieu_de = khoi['name'].map(gia_tu_van_ban)
        g_mo_ta = khoi['description'].map(gia_tu_van_ban)
        dt_tieu_de = khoi['name'].map(dien_tich_tu_van_ban)
        tinh_cu = khoi['province_name'].map(chuan_tinh)
        ket_qua.append(pd.DataFrame({
            'nguon': 1, 'ma_tin': np.nan, 'tieu_de': khoi['name'],
            'loai_bds': khoi['property_type_name'].map(LOAI_1), 'vi_tri': np.nan,
            'tinh': tinh_cu.map(CU_SANG_MOI), 'tinh_cu': tinh_cu, 'dia_gioi_moi': False,
            'quan_huyen': [chuan_quan(q, t) for q, t in zip(khoi['district_name'], tinh_cu)],
            'phuong_xa': khoi['ward_name'].map(chuan_phuong), 'duong': khoi['street_name'], 'du_an': khoi['project_name'],
            'gia_vnd': khoi['price'], 'gia_trich_xuat': g_tieu_de.fillna(g_mo_ta),
            'dien_tich_m2': khoi['area'], 'dien_tich_trich_xuat': dt_tieu_de,
            'mat_tien_m': khoi['frontage_width'], 'chieu_dai_m': khoi['house_depth'], 'duong_vao_m': khoi['road_width'],
            'so_tang': khoi['floor_count'], 'so_phong_ngu': khoi['bedroom_count'], 'so_phong_tam': khoi['bathroom_count'],
            'huong_nha': khoi['house_direction'].map(chuan_huong), 'phap_ly': np.nan, 'noi_that': np.nan,
            'vi_do': np.nan, 'kinh_do': np.nan,
            'ngay_dang': pd.to_datetime(khoi['published_at'], errors='coerce').dt.normalize()}))
        print(f'  đã xử lý {sum(len(x) for x in ket_qua):,} dòng', flush=True)
    return pd.concat(ket_qua, ignore_index=True)

t = time.time()
n1 = doc_nguon1()
print(f'Nguồn 1: {n1.shape} - {time.time() - t:.0f} giây')
n1.head(3)

  đã xử lý 200,000 dòng


  đã xử lý 400,000 dòng


  đã xử lý 600,000 dòng


  đã xử lý 800,000 dòng


  đã xử lý 1,000,000 dòng


Nguồn 1: (1000000, 28) - 123 giây


,nguon,ma_tin,tieu_de,loai_bds,vi_tri,tinh,tinh_cu,dia_gioi_moi,quan_huyen,phuong_xa,duong,du_an,gia_vnd,gia_trich_xuat,dien_tich_m2,dien_tich_trich_xuat,mat_tien_m,chieu_dai_m,duong_vao_m,so_tang,so_phong_ngu,so_phong_tam,huong_nha,phap_ly,noi_that,vi_do,kinh_do,ngay_dang
0,1,NaN,"Bán đất Thanh Đàm, ô tô vào nhà, cách phố 15m, thông số ...",Nhà ở,NaN,Hà Nội,Hà Nội,False,Hoàng Mai,Thanh Trì,Thanh Đàm,NaN,1.085000e+10,1.085000e+10,70.0,70.0,5.0,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-24
1,1,NaN,"Lô góc KĐT Bắc Châu Giang, 97m2, chân 34 tòa chung cư, t...",Biệt thự/Liền kề,NaN,Ninh Bình,Hà Nam,False,Phủ Lý,Lam Hạ,NaN,Sun Urban City Hà Nam,NaN,6.800000e+09,97.0,97.0,9.0,NaN,NaN,4.0,4.0,3.0,NaN,NaN,NaN,NaN,NaN,2025-10-21
2,1,NaN,"Siêu rẻ. Bán đất diện tích 105m2, mặt đường Phan Tôn, tr...",Đất,NaN,Đà Nẵng,Đà Nẵng,False,Ngũ Hành Sơn,Mỹ An,Phan Tôn,NaN,1.950000e+10,1.950000e+10,105.0,105.0,5.0,NaN,5.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-10-16


### Nguồn 2 - giá theo triệu đồng, địa chỉ gộp trong một chuỗi (lẫn địa giới cũ và mới)

In [4]:
LOAI_2 = {'Nhà riêng': 'Nhà ở', 'Đất': 'Đất', 'Căn hộ chung cư': 'Căn hộ', 'Kho, nhà xưởng': 'Kho/Xưởng',
          'Nhà trọ': 'Thương mại khác', 'Khách sạn': 'Thương mại khác', 'Văn phòng': 'Thương mại khác'}

def tach_dia_chi(chuoi):
    phan = [p.strip() for p in str(chuoi).split(',') if p.strip()]
    kq = {'tinh_goc': phan[-1] if phan else np.nan, 'quan': np.nan, 'phuong': np.nan, 'duong': np.nan}
    for p in phan[:-1]:
        t = p.lower()
        if re.match(r'^(phường|xã|thị trấn)\b', t):
            kq['phuong'] = p
        elif re.match(r'^(quận|huyện|thị xã|thành phố|tp\.?)\b', t):
            kq['quan'] = p
        elif re.match(r'^(đường|phố|hẻm|ngõ|ngách|kiệt)\b', t):
            kq['duong'] = re.sub(r'^(đường|phố)\s+', '', p, flags=re.I)
    return kq

def doc_nguon2():
    d = pd.read_csv(os.path.join(RAW, 'nguon2_vn_real_estate_apr_sep_2025.csv'), low_memory=False)
    dc = pd.DataFrame(d['Location'].map(tach_dia_chi).tolist())
    moi = dc['tinh_goc'].str.contains('(Mới)', regex=False).fillna(False).astype(bool)
    tinh_goc = dc['tinh_goc'].map(chuan_tinh)
    tinh_cu = tinh_goc.where(~moi)
    return pd.DataFrame({
        'nguon': 2, 'ma_tin': d['Listing ID'], 'tieu_de': d['Title'],
        'loai_bds': d['Property Type'].map(LOAI_2),
        'vi_tri': d['Position'].map({'Đường chính': 'Mặt phố', 'Trong hẻm': 'Ngõ/Hẻm'}),
        'tinh': np.where(moi, tinh_goc, tinh_cu.map(CU_SANG_MOI)), 'tinh_cu': tinh_cu, 'dia_gioi_moi': moi,
        'quan_huyen': [chuan_quan(q, t) for q, t in zip(dc['quan'], tinh_cu)],
        'phuong_xa': dc['phuong'].map(chuan_phuong), 'duong': dc['duong'], 'du_an': np.nan,
        'gia_vnd': d['Price'] * 1e6,
        'gia_trich_xuat': d['Title'].map(gia_tu_van_ban).fillna(d['Description'].map(gia_tu_van_ban)),
        'dien_tich_m2': d['Area'], 'dien_tich_trich_xuat': d['Title'].map(dien_tich_tu_van_ban),
        'mat_tien_m': d['Width'], 'chieu_dai_m': d['Length'], 'duong_vao_m': d['Alley Width'],
        'so_tang': d['Floors'], 'so_phong_ngu': d['Bedrooms'], 'so_phong_tam': d['Bathrooms'],
        'huong_nha': d['Direction'].map(chuan_huong), 'phap_ly': np.nan, 'noi_that': np.nan,
        'vi_do': d['Latitude'], 'kinh_do': d['Longitude'],
        'ngay_dang': pd.to_datetime(d['Last Updated Date'], format='%d/%m/%Y %H:%M', errors='coerce').dt.normalize()})

n2 = doc_nguon2()
print('Nguồn 2:', n2.shape, '| dòng địa giới mới:', int(n2['dia_gioi_moi'].sum()))
n2.head(3)

Nguồn 2: (236226, 28) | dòng địa giới mới: 75280


,nguon,ma_tin,tieu_de,loai_bds,vi_tri,tinh,tinh_cu,dia_gioi_moi,quan_huyen,phuong_xa,duong,du_an,gia_vnd,gia_trich_xuat,dien_tich_m2,dien_tich_trich_xuat,mat_tien_m,chieu_dai_m,duong_vao_m,so_tang,so_phong_ngu,so_phong_tam,huong_nha,phap_ly,noi_that,vi_do,kinh_do,ngay_dang
0,2,1241101.0,Bán nhà 78.7m² 2.7 tỷ tại Thị trấn Thứ Ba Huyện An Biên,Nhà ở,NaN,An Giang,NaN,True,NaN,An Biên,NaN,NaN,2.700000e+09,2.700000e+09,78.7,78.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.809436,105.056231,2025-09-13
1,2,1208797.0,Bán nhà 192m² 2 tỷ tại Phường Mỹ Hòa Thành phố Long Xuyê...,Nhà ở,NaN,An Giang,NaN,True,NaN,Long Xuyên,NaN,NaN,2.000000e+09,2.000000e+09,192.0,192.0,6.0,NaN,NaN,NaN,3.0,NaN,Tây,NaN,NaN,10.370606,105.412428,2025-09-13
2,2,1182246.0,Bán nhà 34.2m² 600 triệu tại Phường An Hòa Thành phố Rạc...,Nhà ở,NaN,An Giang,NaN,True,NaN,Rạch Giá,NaN,NaN,6.000000e+08,6.000000e+08,34.2,34.2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.983399,105.098417,2025-09-13


### Nguồn 3 - toàn bộ là chuỗi có đơn vị, giá cho theo triệu/m² nên phải nhân với diện tích

In [5]:
DON_VI_GIA_M2 = {'triệu/m²': 1e6, 'tỷ/m²': 1e9, 'đ/m²': 1.0, 'nghìn/m²': 1e3}
LOAI_3 = {'Nhà ngõ, hẻm': ('Nhà ở', 'Ngõ/Hẻm'), 'Nhà mặt phố, mặt tiền': ('Nhà ở', 'Mặt phố'),
          'Nhà biệt thự': ('Biệt thự/Liền kề', np.nan), 'Nhà phố liền kề': ('Biệt thự/Liền kề', np.nan)}

def gia_m2(s):
    if pd.isna(s):
        return np.nan
    dv = re.sub(r'[\d.,\s]', '', str(s))
    return so(s) * DON_VI_GIA_M2[dv] if dv in DON_VI_GIA_M2 else np.nan

def doc_nguon3():
    d = pd.read_csv(os.path.join(RAW, 'nguon3_vn_housing_hanoi_2020.csv'), low_memory=False)
    dt = d['Diện tích'].map(so)
    loai = d['Loại hình nhà ở'].map(LOAI_3)
    return pd.DataFrame({
        'nguon': 3, 'ma_tin': np.nan, 'tieu_de': np.nan,
        'loai_bds': loai.str[0], 'vi_tri': loai.str[1],
        'tinh': 'Hà Nội', 'tinh_cu': 'Hà Nội', 'dia_gioi_moi': False,
        'quan_huyen': d['Quận'].map(lambda q: chuan_quan(q, 'Hà Nội')), 'phuong_xa': d['Huyện'].map(chuan_phuong),
        'duong': d['Địa chỉ'].str.split(',').str[0].str.replace(r'^(đường|phố)\s+', '', regex=True, case=False).str.strip(),
        'du_an': np.nan,
        'gia_vnd': d['Giá/m2'].map(gia_m2) * dt, 'gia_trich_xuat': np.nan,
        'dien_tich_m2': dt, 'dien_tich_trich_xuat': np.nan,
        'mat_tien_m': d['Rộng'].map(so), 'chieu_dai_m': d['Dài'].map(so), 'duong_vao_m': np.nan,
        'so_tang': pd.to_numeric(d['Số tầng'], errors='coerce'), 'so_phong_ngu': d['Số phòng ngủ'].map(so), 'so_phong_tam': np.nan,
        'huong_nha': np.nan, 'phap_ly': d['Giấy tờ pháp lý'].map(chuan_phap_ly), 'noi_that': np.nan,
        'vi_do': np.nan, 'kinh_do': np.nan, 'ngay_dang': pd.to_datetime(d['Ngày'], errors='coerce')})

n3 = doc_nguon3()
print('Nguồn 3:', n3.shape, '| Số tầng không đọc được:', int(pd.to_numeric(pd.read_csv(os.path.join(RAW, 'nguon3_vn_housing_hanoi_2020.csv'), usecols=['Số tầng'])['Số tầng'], errors='coerce').isna().sum()))
n3.head(3)

Nguồn 3: (82497, 28) | Số tầng không đọc được: 46105


,nguon,ma_tin,tieu_de,loai_bds,vi_tri,tinh,tinh_cu,dia_gioi_moi,quan_huyen,phuong_xa,duong,du_an,gia_vnd,gia_trich_xuat,dien_tich_m2,dien_tich_trich_xuat,mat_tien_m,chieu_dai_m,duong_vao_m,so_tang,so_phong_ngu,so_phong_tam,huong_nha,phap_ly,noi_that,vi_do,kinh_do,ngay_dang
0,3,NaN,NaN,Nhà ở,Ngõ/Hẻm,Hà Nội,Hà Nội,False,Cầu Giấy,Nghĩa Đô,Hoàng Quốc Việt,NaN,4.000160e+09,NaN,46.0,NaN,NaN,NaN,NaN,4.0,5.0,NaN,NaN,Có sổ,NaN,NaN,NaN,2020-08-05
1,3,NaN,NaN,Nhà ở,Mặt phố,Hà Nội,Hà Nội,False,Thanh Xuân,Kim Giang,Kim Giang,NaN,4.300140e+09,NaN,37.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,2020-08-05
2,3,NaN,NaN,Nhà ở,Ngõ/Hẻm,Hà Nội,Hà Nội,False,Hai Bà Trưng,Minh Khai,minh khai,NaN,2.600000e+09,NaN,40.0,NaN,4.0,10.0,NaN,4.0,4.0,NaN,NaN,Có sổ,NaN,NaN,NaN,2020-08-05


### Nguồn 4 - giá theo triệu, không có loại bất động sản (suy ra từ tiêu đề), ngày đăng = ngày thu thập − số giờ đã đăng

In [6]:
def doc_nguon4():
    d = pd.read_csv(os.path.join(RAW, 'nguon4_batdongsan_house_buying_2025.csv'), low_memory=False)
    q = d['location'].str.split(',').str[0].str.strip()
    tinh_cu = d['location'].str.split(',').str[-1].map(chuan_tinh)
    return pd.DataFrame({
        'nguon': 4, 'ma_tin': d['id'], 'tieu_de': d['title'],
        'loai_bds': d['title'].map(loai_tu_tieu_de), 'vi_tri': d['frontage'].map({True: 'Mặt phố'}),
        'tinh': tinh_cu.map(CU_SANG_MOI), 'tinh_cu': tinh_cu, 'dia_gioi_moi': False,
        'quan_huyen': [chuan_quan(a, b) for a, b in zip(q, tinh_cu)], 'phuong_xa': np.nan, 'duong': np.nan, 'du_an': np.nan,
        'gia_vnd': d['price_million_vnd'] * 1e6, 'gia_trich_xuat': d['title'].map(gia_tu_van_ban),
        'dien_tich_m2': d['area_m2'], 'dien_tich_trich_xuat': d['title'].map(dien_tich_tu_van_ban),
        'mat_tien_m': np.nan, 'chieu_dai_m': np.nan, 'duong_vao_m': np.nan,
        'so_tang': d['floors'], 'so_phong_ngu': d['bedrooms'], 'so_phong_tam': d['bathrooms'],
        'huong_nha': np.nan, 'phap_ly': np.nan, 'noi_that': np.nan, 'vi_do': np.nan, 'kinh_do': np.nan,
        'ngay_dang': (pd.Timestamp('2025-12-29') - pd.to_timedelta(d['timeline_hours'], unit='h')).dt.normalize()})

n4 = doc_nguon4()
print('Nguồn 4:', n4.shape)
print('Loại BĐS suy ra từ tiêu đề:', n4['loai_bds'].value_counts().to_dict())

Nguồn 4: (66912, 28)
Loại BĐS suy ra từ tiêu đề: {'Nhà ở': 62084, 'Biệt thự/Liền kề': 1546, 'Căn hộ': 1429, 'Đất': 994, 'Kho/Xưởng': 569, 'Shophouse': 290}


### Nguồn 5 - giá theo tỷ, địa chỉ gộp, pháp lý/nội thất lẫn tiếng Anh, không có loại bất động sản

In [7]:
def doc_nguon5():
    d = pd.read_csv(os.path.join(RAW, 'nguon5_vietnam_housing_2024.csv'), low_memory=False)
    phan = d['Address'].str.split(',')
    tinh_cu = phan.str[-1].map(chuan_tinh)
    return pd.DataFrame({
        'nguon': 5, 'ma_tin': np.nan, 'tieu_de': np.nan,
        'loai_bds': np.nan, 'vi_tri': np.nan,
        'tinh': tinh_cu.map(CU_SANG_MOI), 'tinh_cu': tinh_cu, 'dia_gioi_moi': False,
        'quan_huyen': [chuan_quan(a, b) for a, b in zip(phan.str[-2], tinh_cu)],
        'phuong_xa': phan.str[-3].map(chuan_phuong),
        'duong': d['Address'].str.extract(r'(?:Đường|Phố)\s+([^,]+)', expand=False).str.strip(),
        'du_an': d['Address'].str.extract(r'Dự án\s+([^,]+)', expand=False).str.strip(),
        'gia_vnd': d['Price'] * 1e9, 'gia_trich_xuat': np.nan,
        'dien_tich_m2': d['Area'], 'dien_tich_trich_xuat': np.nan,
        'mat_tien_m': d['Frontage'], 'chieu_dai_m': np.nan, 'duong_vao_m': d['Access Road'],
        'so_tang': d['Floors'], 'so_phong_ngu': d['Bedrooms'], 'so_phong_tam': d['Bathrooms'],
        'huong_nha': d['House direction'].map(chuan_huong), 'phap_ly': d['Legal status'].map(chuan_phap_ly),
        'noi_that': d['Furniture state'].map(chuan_noi_that),
        'vi_do': np.nan, 'kinh_do': np.nan, 'ngay_dang': pd.NaT})

n5 = doc_nguon5()
print('Nguồn 5:', n5.shape, '| tỉnh không nhận ra:', n5.loc[n5['tinh'].isna()].shape[0])

Nguồn 5: (30229, 28) | tỉnh không nhận ra: 12


### Nguồn 6 - toàn chuỗi ("15 tỷ", "Thỏa thuận", "63 m²", "5 tầng"), số 0 dùng thay ô trống

In [8]:
LOAI_6 = [(r'^Căn hộ', 'Căn hộ'), (r'^Nhà biệt thự|^Biệt thự', 'Biệt thự/Liền kề'), (r'^Shophouse', 'Shophouse'),
          (r'^Bán đất|^Đất', 'Đất'), (r'^Kho|xưởng', 'Kho/Xưởng'), (r'^Nhà', 'Nhà ở')]

def loai_6(s):
    for mau, ten in LOAI_6:
        if re.search(mau, str(s)):
            return ten
    return 'Thương mại khác'

def gia_nguon6(chuoi, dien_tich):
    t = str(chuoi).lower()
    if '/tháng' in t or '/năm' in t:
        return np.nan
    if '/m²' in t or '/m2' in t:
        return gia_m2(chuoi) * dien_tich
    return gia_tu_chuoi(chuoi)

def doc_nguon6():
    d = pd.read_csv(os.path.join(RAW, 'nguon6_hanoi_houseprice_2024.csv'), low_memory=False)
    dt = d['Area'].map(so_hoac_rong)
    return pd.DataFrame({
        'nguon': 6, 'ma_tin': np.nan, 'tieu_de': d['Title'],
        'loai_bds': d['PostType'].map(loai_6),
        'vi_tri': d['PostType'].map(lambda s: 'Mặt phố' if 'mặt phố' in str(s).lower() else np.nan),
        'tinh': 'Hà Nội', 'tinh_cu': 'Hà Nội', 'dia_gioi_moi': False,
        'quan_huyen': d['District'].map(lambda q: chuan_quan(q, 'Hà Nội')),
        'phuong_xa': d['Address'].str.extract(r'((?:Phường|Xã|Thị trấn)\s+[^,]+)', expand=False).map(chuan_phuong),
        'duong': d['Address'].str.extract(r'(?:Đường|Phố)\s+([^,]+)', expand=False).str.strip(),
        'du_an': d['Address'].str.extract(r'Dự án\s+([^,]+)', expand=False).str.strip(),
        'gia_vnd': [gia_nguon6(g, a) for g, a in zip(d['Price'], dt)], 'gia_trich_xuat': d['Title'].map(gia_tu_van_ban),
        'dien_tich_m2': dt, 'dien_tich_trich_xuat': d['Title'].map(dien_tich_tu_van_ban),
        'mat_tien_m': d['Width_meters'].map(so_hoac_rong), 'chieu_dai_m': np.nan, 'duong_vao_m': d['Entrancewidth'].map(so_hoac_rong),
        'so_tang': d['Floors'].map(so_hoac_rong), 'so_phong_ngu': d['Bedrooms'].map(so_hoac_rong),
        'so_phong_tam': d['Bathrooms'].map(so_hoac_rong),
        'huong_nha': d['Direction'].map(chuan_huong), 'phap_ly': d['Legal'].map(chuan_phap_ly),
        'noi_that': d['Interior'].map(chuan_noi_that),
        'vi_do': np.nan, 'kinh_do': np.nan, 'ngay_dang': pd.to_datetime(d['PostingDate'], format='%d/%m/%Y', errors='coerce')})

n6 = doc_nguon6()
print('Nguồn 6:', n6.shape, '| giá thiếu (Thỏa thuận / tin cho thuê):', int(n6['gia_vnd'].isna().sum()))
n6.head(3)

Nguồn 6: (13535, 28) | giá thiếu (Thỏa thuận / tin cho thuê): 1812


,nguon,ma_tin,tieu_de,loai_bds,vi_tri,tinh,tinh_cu,dia_gioi_moi,quan_huyen,phuong_xa,duong,du_an,gia_vnd,gia_trich_xuat,dien_tich_m2,dien_tich_trich_xuat,mat_tien_m,chieu_dai_m,duong_vao_m,so_tang,so_phong_ngu,so_phong_tam,huong_nha,phap_ly,noi_that,vi_do,kinh_do,ngay_dang
0,6,NaN,Top 10 căn đầu tư tốt nhất tại Vinhome Cổ Loa - mua trực...,Biệt thự/Liền kề,NaN,Hà Nội,Hà Nội,False,Đông Anh,Đông Hội,Cổ Loa,Vinhomes Cổ Loa,1.500000e+10,NaN,63.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-12-16
1,6,NaN,Tổng kho hàng chuyển nhượng vị trí đẹp giá rẻ - giá cho ...,Biệt thự/Liền kề,NaN,Hà Nội,Hà Nội,False,Hoài Đức,Vân Canh,Vành Đai 3.5,An Lạc Green Symphony,2.150000e+10,NaN,90.0,NaN,5.0,NaN,58.0,4.0,NaN,NaN,Tây Nam,Hợp đồng mua bán,NaN,NaN,NaN,2024-12-16
2,6,NaN,CĐT mở bán quỹ căn HOT cuối cùng đẹp nhất The Diamond 25...,Căn hộ,NaN,Hà Nội,Hà Nội,False,Thanh Xuân,Nhân Chính,Lê Văn Lương,NaN,NaN,NaN,107.0,107.0,NaN,NaN,NaN,NaN,3.0,3.0,NaN,Hợp đồng mua bán,NaN,NaN,NaN,2024-12-16


### Ghép 6 nguồn thành một bảng

In [9]:
gop = pd.concat([n[COT] for n in [n1, n2, n3, n4, n5, n6]], ignore_index=True)
for c in ['so_tang', 'so_phong_ngu', 'so_phong_tam', 'mat_tien_m', 'chieu_dai_m', 'duong_vao_m', 'dien_tich_m2', 'gia_vnd']:
    gop[c] = pd.to_numeric(gop[c], errors='coerce')
gop['ma_tin'] = gop['ma_tin'].astype('Int64').astype(str).replace('<NA>', np.nan)
gop['dia_gioi_moi'] = gop['dia_gioi_moi'].astype(bool)
print('Kích thước sau khi ghép:', gop.shape)
tom_tat = gop.groupby('nguon').agg(
    so_dong=('nguon', 'size'),
    thieu_gia=('gia_vnd', lambda s: round(s.isna().mean() * 100, 1)),
    thieu_dien_tich=('dien_tich_m2', lambda s: round(s.isna().mean() * 100, 1)),
    thieu_quan_huyen=('quan_huyen', lambda s: round(s.isna().mean() * 100, 1)),
    thieu_loai=('loai_bds', lambda s: round(s.isna().mean() * 100, 1)),
    tu_ngay=('ngay_dang', 'min'), den_ngay=('ngay_dang', 'max'))
tom_tat

Kích thước sau khi ghép: (1429399, 28)


,so_dong,thieu_gia,thieu_dien_tich,thieu_quan_huyen,thieu_loai,tu_ngay,den_ngay
nguon,,,,,,,
1,1000000,7.8,0.0,2.9,0.0,2025-06-01,2025-12-30
2,236226,0.0,0.1,33.6,0.0,2025-03-17,2025-09-14
3,82497,0.0,0.0,0.0,0.0,2019-08-05,2020-08-05
4,66912,3.1,24.8,0.0,0.0,2022-12-30,2025-12-28
5,30229,0.0,0.0,0.0,100.0,NaT,NaT
6,13535,13.4,0.0,0.0,0.0,2024-10-14,2024-12-21


### Chuẩn hoá mã Unicode và kiểu bỏ dấu
Cùng một địa danh có thể được lưu ở dạng Unicode dựng sẵn (NFC) hoặc tổ hợp (NFD), và được gõ theo kiểu bỏ dấu cũ (Hòa, Thủy) hoặc kiểu mới (Hoà, Thuỷ). Hai chuỗi nhìn giống hệt nhau nhưng máy coi là khác nhau, khiến một phường bị tách thành hai nhóm (sai target encoding, sai trung vị theo khu vực) và bỏ sót tin trùng. Mọi chuỗi văn bản được đưa về NFC và về một kiểu bỏ dấu (kiểu cũ, giống tên tỉnh chính thức như Khánh Hòa, Thanh Hóa).

In [10]:
COT_CHU = ['tieu_de', 'quan_huyen', 'phuong_xa', 'duong', 'du_an']
bang_unicode = []
for c in COT_CHU:
    goc = gop[c].dropna().unique()
    moi = gop[c].map({v: chuan_unicode(v) for v in goc})
    bang_unicode.append({'Cột': c, 'Giá trị khác nhau trước': len(goc), 'Sau chuẩn hoá': moi.nunique(),
                         'Số ô đã đổi': int((gop[c].notna() & (moi != gop[c])).sum())})
    gop[c] = moi
bang_unicode = pd.DataFrame(bang_unicode).set_index('Cột')
bang_unicode.to_csv('../reports/chuan_hoa_unicode.csv', encoding='utf-8-sig')
print('Ví dụ:', [(a, chuan_unicode(a)) for a in ['Trung Hoà', 'Bình Thuỷ', 'Kiến Thuỵ', 'Quỳnh Lưu']])
bang_unicode

Ví dụ: [('Trung Hoà', 'Trung Hòa'), ('Bình Thuỷ', 'Bình Thủy'), ('Kiến Thuỵ', 'Kiến Thụy'), ('Quỳnh Lưu', 'Quỳnh Lưu')]


,Giá trị khác nhau trước,Sau chuẩn hoá,Số ô đã đổi
Cột,,,
tieu_de,1167039,1166975,52289
quan_huyen,661,649,650
phuong_xa,5177,5046,11272
duong,18764,18394,3217
du_an,4859,4856,886


In [11]:
print('Tỉnh (địa giới mới, 34 tỉnh):', gop['tinh'].nunique(), '| không nhận ra:', int(gop['tinh'].isna().sum()))
print('Loại BĐS:', gop['loai_bds'].value_counts(dropna=False).to_dict())
print('Quận 7 sau chuẩn hoá:', gop.loc[gop['quan_huyen'].isin(['Quận 7', '7']), 'quan_huyen'].value_counts().to_dict())
gop.to_parquet(os.path.join(INTERIM, 'gop_6_nguon.parquet'), index=False)
print('Đã lưu data/interim/gop_6_nguon.parquet -', round(os.path.getsize(os.path.join(INTERIM, 'gop_6_nguon.parquet')) / 1e6, 1), 'MB')

Tỉnh (địa giới mới, 34 tỉnh): 34 | không nhận ra: 12
Loại BĐS: {'Nhà ở': 679400, 'Đất': 372097, 'Căn hộ': 242889, 'Biệt thự/Liền kề': 79081, nan: 30261, 'Shophouse': 17188, 'Kho/Xưởng': 5479, 'Thương mại khác': 3004}
Quận 7 sau chuẩn hoá: {'Quận 7': 36320}


Đã lưu data/interim/gop_6_nguon.parquet - 101.6 MB


### Từ điển dữ liệu của bảng chung
Mỗi cột của bảng chung được lấy (và quy đổi) từ cột nào của từng nguồn. "—" nghĩa là nguồn đó không có thông tin này (để trống, xử lý ở các bước sau).

In [12]:
TU_DIEN = [
    ('nguon', 'Mã nguồn dữ liệu (1–6)', 'số nguyên', '', '1', '2', '3', '4', '5', '6'),
    ('ma_tin', 'Mã tin đăng gốc (để phát hiện trùng)', 'chuỗi', '', '—', 'Listing ID', '—', 'id', '—', '—'),
    ('tieu_de', 'Tiêu đề tin đăng', 'chuỗi', '', 'name', 'Title', '—', 'title', '—', 'Title'),
    ('loai_bds', 'Loại BĐS (5 nhóm sau gộp)', 'phân loại', '', 'property_type_name', 'Property Type', 'Loại hình nhà ở', 'suy ra từ title', '— (đoán bằng mô hình)', 'PostType'),
    ('vi_tri', 'Mặt phố hay trong ngõ/hẻm', 'phân loại', '', '—', 'Position', 'Loại hình nhà ở', 'frontage', '—', 'PostType'),
    ('tinh', 'Tỉnh/thành theo địa giới mới (34 tỉnh)', 'phân loại', '', 'province_name', 'Location (cuối)', 'Hà Nội', 'location (cuối)', 'Address (cuối)', 'Hà Nội'),
    ('tinh_cu', 'Tỉnh/thành theo địa giới cũ (63 tỉnh)', 'phân loại', '', 'province_name', 'Location (nếu chưa "(Mới)")', 'Hà Nội', 'location', 'Address', 'Hà Nội'),
    ('quan_huyen', 'Quận/huyện (đã bỏ tiền tố, "7" -> "Quận 7")', 'phân loại', '', 'district_name', 'Location', 'Quận', 'location (đầu)', 'Address', 'District'),
    ('phuong_xa', 'Phường/xã', 'phân loại', '', 'ward_name', 'Location', 'Huyện (ghi nhầm tên cột)', '—', 'Address', 'Address'),
    ('duong', 'Tên đường/phố', 'chuỗi', '', 'street_name', 'Location', 'Địa chỉ', '—', 'Address', 'Address'),
    ('du_an', 'Tên dự án (nếu thuộc dự án)', 'chuỗi', '', 'project_name', '—', '—', '—', 'Address', 'Address'),
    ('gia_vnd', 'GIÁ BÁN – biến mục tiêu', 'số thực', 'VNĐ', 'price (VNĐ)', 'Price × 10⁶', 'Giá/m2 × Diện tích', 'price_million_vnd × 10⁶', 'Price × 10⁹', 'Price (chuỗi "15 tỷ")'),
    ('gia_trich_xuat', 'Giá đọc được từ tiêu đề/mô tả (để khôi phục, kiểm tra)', 'số thực', 'VNĐ', 'name, description', 'Title, Description', '—', 'title', '—', 'Title'),
    ('dien_tich_m2', 'Diện tích đất/sàn', 'số thực', 'm²', 'area', 'Area', 'Diện tích ("46 m²")', 'area_m2', 'Area', 'Area ("63 m²")'),
    ('mat_tien_m', 'Chiều rộng mặt tiền', 'số thực', 'm', 'frontage_width', 'Width', 'Rộng', '—', 'Frontage', 'Width_meters'),
    ('chieu_dai_m', 'Chiều dài lô đất', 'số thực', 'm', 'house_depth', 'Length', 'Dài', '—', '—', '—'),
    ('duong_vao_m', 'Độ rộng đường vào/ngõ', 'số thực', 'm', 'road_width', 'Alley Width', '—', '—', 'Access Road', 'Entrancewidth'),
    ('so_tang', 'Số tầng', 'số thực', 'tầng', 'floor_count', 'Floors', 'Số tầng', 'floors', 'Floors', 'Floors ("5 tầng")'),
    ('so_phong_ngu', 'Số phòng ngủ', 'số thực', 'phòng', 'bedroom_count', 'Bedrooms', 'Số phòng ngủ ("5 phòng")', 'bedrooms', 'Bedrooms', 'Bedrooms'),
    ('so_phong_tam', 'Số phòng tắm/WC', 'số thực', 'phòng', 'bathroom_count', 'Bathrooms', '—', 'bathrooms', 'Bathrooms', 'Bathrooms'),
    ('huong_nha', 'Hướng nhà (8 hướng)', 'phân loại', '', 'house_direction', 'Direction', '—', '—', 'House direction', 'Direction'),
    ('phap_ly', 'Tình trạng pháp lý', 'phân loại', '', '—', '—', 'Giấy tờ pháp lý', '—', 'Legal status (tiếng Anh)', 'Legal'),
    ('noi_that', 'Mức nội thất', 'phân loại', '', '—', '—', '—', '—', 'Furniture state (tiếng Anh)', 'Interior'),
    ('vi_do, kinh_do', 'Toạ độ', 'số thực', 'độ', '—', 'Latitude, Longitude', '—', '—', '—', '—'),
    ('ngay_dang', 'Ngày đăng tin', 'ngày', '', 'published_at', 'Last Updated Date', 'Ngày', '29/12/2025 − timeline_hours', '— (ước lượng 2024)', 'PostingDate'),
]
tu_dien = pd.DataFrame(TU_DIEN, columns=['Cột', 'Ý nghĩa', 'Kiểu', 'Đơn vị', 'Nguồn 1', 'Nguồn 2', 'Nguồn 3', 'Nguồn 4', 'Nguồn 5', 'Nguồn 6'])
os.makedirs('../reports', exist_ok=True)
tu_dien.to_csv('../reports/tu_dien_du_lieu.csv', index=False, encoding='utf-8-sig')
tu_dien

,Cột,Ý nghĩa,Kiểu,Đơn vị,Nguồn 1,Nguồn 2,Nguồn 3,Nguồn 4,Nguồn 5,Nguồn 6
0,nguon,Mã nguồn dữ liệu (1–6),số nguyên,,1,2,3,4,5,6
1,ma_tin,Mã tin đăng gốc (để phát hiện trùng),chuỗi,,—,Listing ID,—,id,—,—
2,tieu_de,Tiêu đề tin đăng,chuỗi,,name,Title,—,title,—,Title
3,loai_bds,Loại BĐS (5 nhóm sau gộp),phân loại,,property_type_name,Property Type,Loại hình nhà ở,suy ra từ title,— (đoán bằng mô hình),PostType
4,vi_tri,Mặt phố hay trong ngõ/hẻm,phân loại,,—,Position,Loại hình nhà ở,frontage,—,PostType
5,tinh,Tỉnh/thành theo địa giới mới (34 tỉnh),phân loại,,province_name,Location (cuối),Hà Nội,location (cuối),Address (cuối),Hà Nội
6,tinh_cu,Tỉnh/thành theo địa giới cũ (63 tỉnh),phân loại,,province_name,"Location (nếu chưa ""(Mới)"")",Hà Nội,location,Address,Hà Nội
7,quan_huyen,"Quận/huyện (đã bỏ tiền tố, ""7"" -> ""Quận 7"")",phân loại,,district_name,Location,Quận,location (đầu),Address,District
8,phuong_xa,Phường/xã,phân loại,,ward_name,Location,Huyện (ghi nhầm tên cột),—,Address,Address
9,duong,Tên đường/phố,chuỗi,,street_name,Location,Địa chỉ,—,Address,Address
